# 04 — Generation e Prompt Engineering para RAG

O prompt e a interface entre o contexto recuperado e a resposta do LLM.
Um bom prompt RAG:
1. Instrui o LLM a responder APENAS com base no contexto
2. Inclui o contexto de forma clara
3. Pede para o LLM citar fontes quando possivel
4. Define o tom e formato da resposta

**Prerequisito:** Ollama rodando com llama3.2

In [ ]:
import sys
sys.path.insert(0, '..')

import ollama
import httpx

try:
    r = httpx.get('http://localhost:11434/api/tags')
    modelos = [m['name'] for m in r.json().get('models', [])]
    LLM = 'llama3.2' if any('llama3.2' in m for m in modelos) else (modelos[0] if modelos else None)
    print(f'LLM: {LLM}')
    print(f'Todos: {modelos}')
except Exception as e:
    LLM = None
    print(f'Ollama offline: {e}')

# Contexto de exemplo para testes
contexto_exemplo = """
[Fonte: vector_databases_overview.md]
HNSW (Hierarchical Navigable Small World) e o algoritmo de indexacao mais popular para ANN.
Os parametros principais sao:
- m: numero de conexoes por no (default 16)
- ef_construct: candidatos durante construcao (default 100)
- ef: candidatos durante busca

[Fonte: rag_fundamentals.md]
Para RAG de producao, recomenda-se usar int8 Scalar Quantization no Qdrant.
Isso reduz memoria em 75% com apenas 1-3% de perda de recall.
O parametro rescore=True recupera a precisao original.
"""

print('Pronto!')

## 4.1 Prompts: Basico vs Avancado

In [ ]:
PROMPTS = {
    'basico': """
Contexto: {contexto}

Pergunta: {pergunta}

Resposta:""",

    'strict_grounding': """
Voce e um assistente tecnico. Responda a pergunta baseado APENAS no contexto abaixo.
Se a resposta nao estiver no contexto, diga exatamente: "Nao encontrei essa informacao no contexto fornecido."
Nao use conhecimento externo.

Contexto:
{contexto}

Pergunta: {pergunta}

Resposta:""",

    'with_citation': """
Voce e um assistente tecnico especializado. Responda a pergunta usando APENAS o contexto abaixo.
Cite a fonte entre colchetes quando usar uma informacao especifica.
Se a resposta nao estiver no contexto, diga: "Informacao nao disponivel no contexto."

Contexto:
{contexto}

Pergunta: {pergunta}

Resposta com citacoes:""",

    'structured': """
Voce e um assistente tecnico. Use apenas o contexto fornecido para responder.

Contexto:
{contexto}

Pergunta: {pergunta}

Responda no formato:
**Resposta direta:** (1-2 frases)
**Detalhes:** (expandir se necessario)
**Fonte:** (qual parte do contexto foi usada)""",
}

def gerar(pergunta, template_nome, contexto=contexto_exemplo):
    if not LLM:
        return '[Ollama offline]'
    prompt = PROMPTS[template_nome].format(contexto=contexto, pergunta=pergunta)
    response = ollama.chat(
        model=LLM,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']

print('Templates definidos:', list(PROMPTS.keys()))

In [ ]:
pergunta = 'Quais sao os parametros do HNSW e o que cada um faz?'

print(f'Pergunta: {pergunta}')
print('='*60)

for nome in PROMPTS:
    print(f'\n--- Template: {nome} ---')
    resposta = gerar(pergunta, nome)
    print(resposta[:500])
    if len(resposta) > 500:
        print('...(truncado)')

## 4.2 Alucinacao: Quando o LLM vai alem do contexto

In [ ]:
# Testar com pergunta cuja resposta NAO esta no contexto
pergunta_sem_resposta = 'Qual e a melhor receita de bolo de chocolate?'

print(f'Pergunta (fora do contexto): {pergunta_sem_resposta}')
print('='*60)

print('\n--- Prompt BASICO (propenso a alucinar) ---')
resposta_basica = gerar(pergunta_sem_resposta, 'basico')
print(resposta_basica[:300])

print('\n--- Prompt STRICT GROUNDING (deve recusar) ---')
resposta_strict = gerar(pergunta_sem_resposta, 'strict_grounding')
print(resposta_strict[:300])

print('\nLicao: Instrucoes explicitas reduzem alucinacao!')

## 4.3 Streaming de Respostas

In [ ]:
if LLM:
    print('Resposta com streaming:')
    print('-'*40)
    
    prompt = PROMPTS['with_citation'].format(
        contexto=contexto_exemplo,
        pergunta='Como reduzir o uso de memoria do Qdrant em producao?'
    )
    
    stream = ollama.chat(
        model=LLM,
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,
    )
    
    for chunk in stream:
        print(chunk['message']['content'], end='', flush=True)
    print('\n' + '-'*40)
else:
    print('Ollama offline — streaming nao disponivel')

## 4.4 Context Stuffing: Qual e o limite?

Cada modelo tem um limite de contexto. Com muitos chunks, o LLM pode se perder.

In [ ]:
# Demonstrar como o tamanho do contexto afeta a resposta
chunks_grandes = [
    'Chunk 1: HNSW usa grafo hierarquico para busca eficiente.',
    'Chunk 2: BM25 e algoritmo de ranking por frequencia de termos.',
    'Chunk 3: Embeddings capturam semantica em espaco vetorial denso.',
    'Chunk 4: Transformers usam self-attention para relacoes globais.',
    'Chunk 5: Scalar quantization reduz memoria em 4x com int8.',
    # Chunks distratores (irrelevantes para a query)
    'Chunk 6: Python e uma linguagem de programacao interpretada.',
    'Chunk 7: Docker containeriza aplicacoes para deploy consistente.',
    'Chunk 8: Git e um sistema de controle de versao distribuido.',
    'Chunk 9: Redis e um banco de dados in-memory de alta performance.',
    'Chunk 10: React e uma biblioteca JavaScript para UIs.',
]

def build_context(n_chunks):
    return '\n'.join(chunks_grandes[:n_chunks])

pergunta_foco = 'Como funciona a busca vetorial eficiente?'

print('Impacto do numero de chunks na resposta:')
print('='*60)

for n in [2, 5, 10]:
    ctx = build_context(n)
    if LLM:
        prompt = PROMPTS['strict_grounding'].format(contexto=ctx, pergunta=pergunta_foco)
        resp = ollama.chat(model=LLM, messages=[{'role': 'user', 'content': prompt}])['message']['content']
    else:
        resp = '[Ollama offline]'
    
    print(f'\n--- {n} chunks ({n - min(n, 5)} distratores) ---')
    print(resp[:200])

## Boas Praticas de Prompt para RAG

1. **Seja explicito sobre grounding**: "responda APENAS com base no contexto"
2. **Defina comportamento para ausencia de info**: "se nao souber, diga X"
3. **Peca citacoes**: aumenta faithfulness e permite verificacao
4. **Controle o formato**: estrutura a resposta para o seu caso de uso
5. **Use separadores claros**: `[Fonte: ...]` ou `---` entre chunks
6. **Limite o contexto**: 3-5 chunks relevantes > 10 chunks misturados
7. **Re-rank antes de incluir**: use um cross-encoder para filtrar

## Proximo
- [Modulo 04 — Arquiteturas RAG](../04_rag_architectures/README.md)